# SquirrelBGone — YOLOv8n Training
Trains a lightweight YOLOv8n model on the Warren Wiens squirrel/bird dataset.
Output: `best.pt` — ready to run locally on the Raspberry Pi via ultralytics.

**Before running:** Runtime → Change runtime type → T4 GPU

## Step 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none — go to Runtime > Change runtime type > T4 GPU'}")

## Step 2 — Install dependencies

In [ ]:
%pip install -q ultralytics roboflow
print("Done.")

## Step 3 — Download dataset from Roboflow Universe

Uses the Warren Wiens Squirrel Detector 1.1 dataset (5,102 images, open source).
Classes: `squirrel`, `bird`, `cat`, `dog`, `raccoon`, `skunk`, `deer`.

Paste your Roboflow API key below — free account, no credit card needed.
Get it at: https://app.roboflow.com/settings/api

In [ ]:
from roboflow import Roboflow
import os

# Paste your key here
ROBOFLOW_API_KEY = "rf_your_key_here"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("warren-wiens-d0d4p").project("squirrel-detector-1.1")

# Download in YOLOv8 format — includes train/val/test split and data.yaml
dataset = project.version(1).download("yolov8")

print(f"\nDataset location: {dataset.location}")
print(f"data.yaml: {dataset.location}/data.yaml")

## Step 4 — Inspect the dataset
Quick sanity check before training.

In [ ]:
import yaml
from pathlib import Path

data_yaml = Path(dataset.location) / "data.yaml"
with open(data_yaml) as f:
    data = yaml.safe_load(f)

print("Classes:", data["names"])
print("Num classes:", data["nc"])

# Count images in each split
for split in ["train", "valid", "test"]:
    img_dir = Path(dataset.location) / split / "images"
    if img_dir.exists():
        count = len(list(img_dir.glob("*.*")))
        print(f"{split}: {count} images")

## Step 5 — Train YOLOv8n

- `yolov8n.pt` = nano model, ideal for Pi CPU inference
- 50 epochs is enough for a pre-annotated dataset this size
- Expect ~20-30 minutes on a T4 GPU

Results will be saved to `runs/detect/squirrelbgone/`

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # start from nano pretrained on COCO

results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=16,          # safe for T4; increase to 32 if no OOM error
    name="squirrelbgone",
    patience=10,       # early stop if no improvement for 10 epochs
    save=True,
    plots=True,
    device=0,          # GPU
    workers=2,
)

print("\nTraining complete.")
print(f"Best weights: {results.save_dir}/weights/best.pt")

## Step 6 — Evaluate on test set

In [ ]:
best_model = YOLO(f"{results.save_dir}/weights/best.pt")

metrics = best_model.val(
    data=str(data_yaml),
    split="test",
)

print(f"\nmAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

# Per-class breakdown
print("\nPer-class AP50:")
for name, ap in zip(data["names"], metrics.box.ap50):
    print(f"  {name:<12} {ap:.3f}")

## Step 7 — Copy best.pt to a convenient location and download it

Download this file, then add it to your git repo under `models/best.pt`.

In [ ]:
import shutil
from google.colab import files

best_pt = f"{results.save_dir}/weights/best.pt"
shutil.copy(best_pt, "/content/squirrelbgone_best.pt")

print(f"Model size: {Path(best_pt).stat().st_size / 1e6:.1f} MB")
print("Downloading...")
files.download("/content/squirrelbgone_best.pt")

## Step 8 — Quick smoke test (optional)
Run a prediction on a sample image to visually confirm the model works before you leave Colab.

In [ ]:
from pathlib import Path
import glob
from IPython.display import Image, display

# Grab a test image
test_images = glob.glob(f"{dataset.location}/test/images/*.*")
if test_images:
    sample = test_images[0]
    result = best_model.predict(sample, conf=0.4, save=True, project="/content", name="smoke_test")
    
    # Show the annotated result
    annotated = glob.glob("/content/smoke_test/*.jpg") + glob.glob("/content/smoke_test/*.png")
    if annotated:
        display(Image(annotated[0], width=640))
    
    for r in result:
        for box in r.boxes:
            cls = data["names"][int(box.cls)]
            conf = float(box.conf)
            print(f"  {cls}: {conf:.2f}")
else:
    print("No test images found — skipping smoke test.")

## What to do with best.pt

1. Move the downloaded file into your repo: `models/squirrelbgone_best.pt`
2. Commit and push to GitHub
3. On the Pi: `git pull`
4. The detect script will load it with:
   ```python
   from ultralytics import YOLO
   model = YOLO('models/squirrelbgone_best.pt')
   ```
   No API calls, no internet required at runtime, no cost.